# Watt The Hack — Local Playtester

Notebook version: 3

Thin wrapper around the local playtest CLI for people who want inline plots. Everything here runs against your local engine and scenarios — no Colab uploads, no zip files.

## One-time setup (from the repo root, in your shell)

```
python -m pip install -e ".[playtest]"
python -m pip install pandas   # optional, for inline dataframe display below
```

## Workflow

1. Edit `playtester/baseline_controllers/my_controller.py` (or any `.py` with a `Strategy` class / `controller` function).
2. Run the cell below to evaluate it against any scenario.
3. Inspect the per-step CSV inline, or open the auto-generated PNGs in `runs/<scenario>_<timestamp>/`.

In [ ]:
from pathlib import Path
import sys

# Make the repo importable if the notebook was launched from notebooks/.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from watt_the_hack.playtest import run_playtest

CONTROLLER = REPO_ROOT / "playtester" / "baseline_controllers" / "my_controller.py"
SCENARIO_ID = "duck_curve"     # try: t1_welcome, duck_curve, melbourne_cold_winter, storm_front

result = run_playtest(
    controller_path=CONTROLLER,
    scenario_id=SCENARIO_ID,
    plots=True,
)
out_dir = Path(result["out_dir"])

## Inline plots

The cell above already wrote PNGs to `out_dir`. Re-draw inline below for quick iteration without leaving the notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(out_dir / "steps.csv")
df.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

axes[0].plot(df["time_hours"], df["soc"], color="tab:blue")
axes[0].set_ylabel("SOC")
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)

axes[1].plot(df["time_hours"], df["demand"], label="demand", color="tab:orange")
axes[1].plot(df["time_hours"], df["solar"], label="solar", color="tab:green")
axes[1].plot(df["time_hours"], df["net_grid_power_mw"], label="net grid", color="tab:gray")
axes[1].set_ylabel("MW")
axes[1].legend(loc="upper right")
axes[1].grid(True, alpha=0.3)

axes[2].plot(df["time_hours"], df["cum_cost"], color="tab:red")
axes[2].set_ylabel("cumulative cost ($)")
axes[2].set_xlabel("hours")
axes[2].grid(True, alpha=0.3)

fig.suptitle(f"{SCENARIO_ID} — final score ${result['metrics']['final_score']:.2f}")
fig.tight_layout()
plt.show()

## Cost breakdown

Lowest cost wins. Negative components are revenue.

In [ ]:
breakdown = {k: v for k, v in result["breakdown"].items() if k != "total"}
pd.Series(breakdown).sort_values(key=lambda s: -s.abs()).to_frame("$")

## Iterate

Edit `playtester/baseline_controllers/my_controller.py`, then re-run the first code cell. The CLI auto-generates a fresh `runs/<scenario>_<timestamp>/` each run so you can `diff` between iterations.

## Other ways to drive the harness

```
python -m watt_the_hack.playtest --list-scenarios
python -m watt_the_hack.playtest playtester/baseline_controllers/my_controller.py --scenario storm_front
python -m watt_the_hack.playtest playtester/baseline_controllers/my_controller.py --scenario duck_curve --steps 50 --no-plots
```